In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [2]:
spark = SparkSession.builder\
    .master("spark://spark-master:7077")\
    .appName("Classification")\
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/08 19:32:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/08 19:32:44 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
data = spark.read.csv("s3a://bronze/cliente-classificacao", sep = ',', header=True, inferSchema=True)

In [4]:
data.printSchema()

root
 |-- id: integer (nullable = true)
 |-- Churn: string (nullable = true)
 |-- Mais65anos: integer (nullable = true)
 |-- Conjuge: string (nullable = true)
 |-- Dependentes: string (nullable = true)
 |-- MesesDeContrato: integer (nullable = true)
 |-- TelefoneFixo: string (nullable = true)
 |-- MaisDeUmaLinhaTelefonica: string (nullable = true)
 |-- Internet: string (nullable = true)
 |-- SegurancaOnline: string (nullable = true)
 |-- BackupOnline: string (nullable = true)
 |-- SeguroDispositivo: string (nullable = true)
 |-- SuporteTecnico: string (nullable = true)
 |-- TVaCabo: string (nullable = true)
 |-- StreamingFilmes: string (nullable = true)
 |-- TipoContrato: string (nullable = true)
 |-- ContaCorreio: string (nullable = true)
 |-- MetodoPagamento: string (nullable = true)
 |-- MesesCobrados: double (nullable = true)



In [5]:
data.show(5)

+---+-----+----------+-------+-----------+---------------+------------+------------------------+-----------+---------------+------------+-----------------+--------------+-------+---------------+------------+------------+----------------+-------------+
| id|Churn|Mais65anos|Conjuge|Dependentes|MesesDeContrato|TelefoneFixo|MaisDeUmaLinhaTelefonica|   Internet|SegurancaOnline|BackupOnline|SeguroDispositivo|SuporteTecnico|TVaCabo|StreamingFilmes|TipoContrato|ContaCorreio| MetodoPagamento|MesesCobrados|
+---+-----+----------+-------+-----------+---------------+------------+------------------------+-----------+---------------+------------+-----------------+--------------+-------+---------------+------------+------------+----------------+-------------+
|  0|  Nao|         0|    Sim|        Nao|              1|         Nao|    SemServicoTelefonico|        DSL|            Nao|         Sim|              Nao|           Nao|    Nao|            Nao| Mensalmente|         Sim|BoletoEletronico|       

In [6]:
for c in data:
    data.groupBy(c)\
    .agg(F.count("*").alias("QTD"))\
    .show()

+----+---+
|  id|QTD|
+----+---+
| 148|  1|
| 463|  1|
| 471|  1|
| 496|  1|
| 833|  1|
|1088|  1|
|1238|  1|
|1342|  1|
|1580|  1|
|1591|  1|
|1645|  1|
|1829|  1|
|1959|  1|
|2122|  1|
|2142|  1|
|2366|  1|
|2659|  1|
|2866|  1|
|3175|  1|
|3749|  1|
+----+---+
only showing top 20 rows

+-----+----+
|Churn| QTD|
+-----+----+
|  Sim|5174|
|  Nao|5174|
+-----+----+

+----------+----+
|Mais65anos| QTD|
+----------+----+
|         1|1459|
|         0|8889|
+----------+----+

+-------+----+
|Conjuge| QTD|
+-------+----+
|    Sim|4481|
|    Nao|5867|
+-------+----+

+-----------+----+
|Dependentes| QTD|
+-----------+----+
|        Sim|2477|
|        Nao|7871|
+-----------+----+

+---------------+----+
|MesesDeContrato| QTD|
+---------------+----+
|             31|  90|
|             65|  88|
|             53|  88|
|             34|  82|
|             28|  79|
|             27| 101|
|             26| 104|
|             44|  72|
|             12| 186|
|             22| 124|
|             47|

In [7]:
bollcolumns = [
    'Churn',
    'Conjuge',
    'Dependentes',
    'TelefoneFixo',
    'MaisDeUmaLinhaTelefonica',
    'SegurancaOnline',
    'BackupOnline',
    'SeguroDispositivo',
    'SuporteTecnico',
    'TVaCabo',
    'StreamingFilmes',
    'ContaCorreio'
]

transformed = [
    F.when(F.col(c) == 'Sim', 1).otherwise(0).alias(c) for c in bollcolumns
]

df_final = data.select(
    *[c for c in data.columns if c not in bollcolumns],
    *transformed 
)

df_final.show()

+---+----------+---------------+-----------+------------+----------------+-------------+-----+-------+-----------+------------+------------------------+---------------+------------+-----------------+--------------+-------+---------------+------------+
| id|Mais65anos|MesesDeContrato|   Internet|TipoContrato| MetodoPagamento|MesesCobrados|Churn|Conjuge|Dependentes|TelefoneFixo|MaisDeUmaLinhaTelefonica|SegurancaOnline|BackupOnline|SeguroDispositivo|SuporteTecnico|TVaCabo|StreamingFilmes|ContaCorreio|
+---+----------+---------------+-----------+------------+----------------+-------------+-----+-------+-----------+------------+------------------------+---------------+------------+-----------------+--------------+-------+---------------+------------+
|  0|         0|              1|        DSL| Mensalmente|BoletoEletronico|        29.85|    0|      1|          0|           0|                       0|              0|           1|                0|             0|      0|              0|      

In [8]:
colunas_dummies = ['Internet', 'TipoContrato', 'MetodoPagamento']

for c in colunas_dummies:
    categorias = [row[c] for row in df_final.select(c).distinct().collect()]
    for cat in categorias:
        df_final = df_final.withColumn(
            f"{c}_{cat}", 
            F.when(F.col(c) == cat, 1).otherwise(0)
        )

df_final = df_final.drop(*colunas_dummies)

df_final.show()

+---+----------+---------------+-------------+-----+-------+-----------+------------+------------------------+---------------+------------+-----------------+--------------+-------+---------------+------------+--------------------+------------+------------+------------------+------------------------+---------------------+--------------------------------+-----------------------------+-----------------------------+----------------------+
| id|Mais65anos|MesesDeContrato|MesesCobrados|Churn|Conjuge|Dependentes|TelefoneFixo|MaisDeUmaLinhaTelefonica|SegurancaOnline|BackupOnline|SeguroDispositivo|SuporteTecnico|TVaCabo|StreamingFilmes|ContaCorreio|Internet_FibraOptica|Internet_Nao|Internet_DSL|TipoContrato_UmAno|TipoContrato_Mensalmente|TipoContrato_DoisAnos|MetodoPagamento_BoletoEletronico|MetodoPagamento_CartaoCredito|MetodoPagamento_DebitoEmConta|MetodoPagamento_Boleto|
+---+----------+---------------+-------------+-----+-------+-----------+------------+------------------------+------------

In [9]:
df_final = df_final.withColumnRenamed('Churn', 'label')

In [10]:
df_features = df_final.columns
df_features.remove('label')
df_features.remove('id')

In [11]:
assembler = VectorAssembler(inputCols=df_features, outputCol='features')

In [12]:
df_prep = assembler.transform(df_final).select('features', 'label')

In [13]:
df_prep.show(5, truncate=False)

+--------------------------------------------------------------------+-----+
|features                                                            |label|
+--------------------------------------------------------------------+-----+
|(24,[1,2,3,8,13,16,18,20],[1.0,29.85,1.0,1.0,1.0,1.0,1.0,1.0])      |0    |
|(24,[1,2,5,7,9,16,17,23],[34.0,56.95,1.0,1.0,1.0,1.0,1.0,1.0])      |0    |
|(24,[1,2,5,7,8,13,16,18,23],[2.0,53.85,1.0,1.0,1.0,1.0,1.0,1.0,1.0])|1    |
|(24,[1,2,7,9,10,16,17,22],[45.0,42.3,1.0,1.0,1.0,1.0,1.0,1.0])      |0    |
|(24,[1,2,5,13,14,18,20],[2.0,70.7,1.0,1.0,1.0,1.0,1.0])             |1    |
+--------------------------------------------------------------------+-----+
only showing top 5 rows



In [14]:
seed= 101

In [15]:
treino, teste = df_prep.randomSplit([0.7, 0.3], seed=seed)

In [24]:
rfc = RandomForestClassifier(seed=seed)
modelo_rfc = rfc.fit(treino)

In [25]:
prev_rfc_teste = modelo_rfc.transform(teste)
prev_rfc_treino = modelo_rfc.transform(treino)

In [26]:
def calcula_mostra_matriz_confusao(tipo ,df_transform_modelo, normalize=False, percentage=True):
    tp = df_transform_modelo.where((F.col('label') == 1) & (F.col('prediction') == 1)).count()
    tn = df_transform_modelo.where((F.col('label') == 0) & (F.col('prediction') == 0)).count()
    fp = df_transform_modelo.where((F.col('label') == 0) & (F.col('prediction') == 1)).count()
    fn = df_transform_modelo.where((F.col('label') == 1) & (F.col('prediction') == 0)).count()
    
    valorP = tp + fn if normalize else 1
    valorN = fp + tn if normalize else 1
    
    if percentage and normalize:
        valorP /= 100
        valorN /= 100
    
    tp_val = int(tp / valorP)
    fn_val = int(fn / valorP)
    fp_val = int(fp / valorN)
    tn_val = int(tn / valorN)
    
    print(f"\nMatriz de Confusão - {tipo}")
    print("                 Predito")
    print("            |  Churn  | Não-Churn |")
    print("------------*---------*-----------*")
    print(f" Real Churn |{tp_val:^9}|{fn_val:^11}|")
    print("------------*---------*-----------*")
    print(f" Não-Churn  |{fp_val:^9}|{tn_val:^11}|")
    print("------------*---------*-----------*")

In [27]:
calcula_mostra_matriz_confusao("Teste",prev_rfc_teste, normalize=False)
calcula_mostra_matriz_confusao("Treino",prev_rfc_treino, normalize=False)


Matriz de Confusão - Teste
                 Predito
            |  Churn  | Não-Churn |
------------*---------*-----------*
 Real Churn |  1294   |    286    |
------------*---------*-----------*
 Não-Churn  |   397   |   1165    |
------------*---------*-----------*

Matriz de Confusão - Treino
                 Predito
            |  Churn  | Não-Churn |
------------*---------*-----------*
 Real Churn |  2955   |    639    |
------------*---------*-----------*
 Não-Churn  |   900   |   2712    |
------------*---------*-----------*


In [28]:
evaluator = MulticlassClassificationEvaluator()

In [30]:
def avaliar_modelo(predicoes, label_col="label", prediction_col="prediction", classe=1):
    evaluator = MulticlassClassificationEvaluator()
    
    acuracia = evaluator.evaluate(predicoes, {evaluator.metricName: "accuracy"})
    precisao = evaluator.evaluate(predicoes, {evaluator.metricName: "precisionByLabel", evaluator.metricLabel: classe})
    recall   = evaluator.evaluate(predicoes, {evaluator.metricName: "recallByLabel", evaluator.metricLabel: classe})
    f1       = evaluator.evaluate(predicoes, {evaluator.metricName: "fMeasureByLabel", evaluator.metricLabel: classe})
    
    print(f"Acurácia: {acuracia:.4f}")
    print(f"Precisão: {precisao:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1: {f1:.4f}\n")
    
    return {"acuracia": acuracia, "precisao": precisao, "recall": recall, "f1": f1}


In [31]:
resultados = avaliar_modelo(prev_rfc_teste)
resultados = avaliar_modelo(prev_rfc_treino)

Acurácia: 0.7826
Precisão: 0.7652
Recall: 0.8190
F1: 0.7912

Acurácia: 0.7864
Precisão: 0.7665
Recall: 0.8222
F1: 0.7934

